# ML-07 — Baseline Action Score and Top-10 Review

This notebook builds a simple, auditable baseline for the **Search visibility / content decline detection** lane.

**Decision:** prioritize content pages for editor review using only information available at the decision cutoff.

**Feature window:** February 2026  
**Decision cutoff:** 2026-02-28  
**Future label/outcome window:** March 2026 (used only for signal audit, never as a scoring feature)

The baseline deliberately stays simple so that the Week-5 model has a clear benchmark to beat.

> Important: the signal-audit outcome is used only to check whether a candidate signal is directionally useful. It is not included in the baseline score.


## 0. Setup and data contract

The baseline uses two decision-time signals:

1. `days_since_last_update` — linked to the FlyRank refresh/staleness flag.
2. `impressions_90d` — linked to the volume / quick-win prioritization idea.

The score must not use future-window data or fields that define the proxy outcome, such as `trend_direction` or `trend_pct`.


In [2]:
import os
import subprocess
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

# Work from the repository when the notebook is inside work/notebooks/.
# If opened directly in Colab, clone the user's repository once.
REPO_URL = "https://github.com/enoughsudhanshu/flyrank-ml-internship-starter.git"
REPO_DIR = "/content/flyrank-ml-internship-starter"

if os.path.exists("../../data/raw/content_refresh_anonymized.csv"):
    DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"
elif os.path.exists(os.path.join(REPO_DIR, "data", "raw", "content_refresh_anonymized.csv")):
    DATA_PATH = os.path.join(REPO_DIR, "data", "raw", "content_refresh_anonymized.csv")
else:
    if not os.path.exists(REPO_DIR):
        result = subprocess.run(
            ["git", "clone", "-q", REPO_URL, REPO_DIR],
            capture_output=True,
            text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(
                "Could not clone the repository. Check that the repository is accessible.\n"
                + result.stderr
            )
    DATA_PATH = os.path.join(
        REPO_DIR, "data", "raw", "content_refresh_anonymized.csv"
    )

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Path:", DATA_PATH)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumns:")
print(df.columns.tolist())


Dataset loaded successfully.
Path: /content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv
Rows: 30000
Columns: 44

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [3]:
# Required decision-time signals for this baseline.
REQUIRED = ["days_since_last_update", "impressions_90d"]

missing = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

# Create a proxy only for the signal audit if the starter dataset provides it.
# IMPORTANT: this proxy is never used in the baseline score.
if "trend_direction" in df.columns:
    df["is_declining_proxy"] = (
        df["trend_direction"].astype(str).str.lower().eq("down")
    ).astype(int)
    print("Proxy outcome available for audit: yes")
    print("Proxy positive rate:", round(df["is_declining_proxy"].mean(), 3))
else:
    df["is_declining_proxy"] = np.nan
    print("Proxy outcome available for audit: no")
    print("Signal tables will still be produced, but outcome-rate comparison is unavailable.")


Proxy outcome available for audit: yes
Proxy positive rate: 0.542


## 1. Signal checks

### Signal 1 — `days_since_last_update`

This is the flag-linked signal. The FlyRank session connected refresh/staleness flags with how long content has gone without an update.

We bucket the signal and print `n` so the verdict is based on observed group sizes rather than an unsupported threshold.


In [4]:
# Signal 1 bucket table: staleness.
staleness_bins = [-np.inf, 30, 90, 180, np.inf]
staleness_labels = ["0-30 days", "31-90 days", "91-180 days", "181+ days"]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=staleness_bins,
    labels=staleness_labels,
    include_lowest=True
)

if df["is_declining_proxy"].notna().any():
    staleness_table = (
        df.groupby("staleness_bucket", observed=False)
          .agg(
              n=("days_since_last_update", "size"),
              decline_rate=("is_declining_proxy", "mean"),
              median_impressions_90d=("impressions_90d", "median")
          )
          .reset_index()
    )
else:
    staleness_table = (
        df.groupby("staleness_bucket", observed=False)
          .agg(
              n=("days_since_last_update", "size"),
              median_impressions_90d=("impressions_90d", "median")
          )
          .reset_index()
    )

display(staleness_table)
print("\nSignal: days_since_last_update")
print("The table above is the evidence used for the staleness verdict.")


,staleness_bucket,n,decline_rate,median_impressions_90d
0,0-30 days,20480,0.511377,470.0
1,31-90 days,175,0.588571,510.0
2,91-180 days,9171,0.611057,1692.0
3,181+ days,174,0.471264,15.5



Signal: days_since_last_update
The table above is the evidence used for the staleness verdict.


### Staleness verdict

**Verdict: CONFIRMED**

The baseline treats staleness as a useful directional prioritization signal because it is directly linked to the FlyRank refresh/staleness flag. The bucket table above is the empirical check; the score below does not use any future outcome field.

This is a decision-support claim, not a causal claim: being stale does not prove that refreshing a page will improve performance.


### Signal 2 — `impressions_90d`

This is the volume signal. The idea is that an editor has a potentially more useful refresh opportunity when a page already has measurable search visibility.

We use quantile buckets so the check does not depend on an arbitrary raw-impression cutoff.


In [5]:
# Signal 2 bucket table: search volume.
df["volume_bucket"] = pd.qcut(
    df["impressions_90d"].rank(method="first"),
    q=4,
    labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"]
)

if df["is_declining_proxy"].notna().any():
    volume_table = (
        df.groupby("volume_bucket", observed=False)
          .agg(
              n=("impressions_90d", "size"),
              median_impressions_90d=("impressions_90d", "median"),
              decline_rate=("is_declining_proxy", "mean")
          )
          .reset_index()
    )
else:
    volume_table = (
        df.groupby("volume_bucket", observed=False)
          .agg(
              n=("impressions_90d", "size"),
              median_impressions_90d=("impressions_90d", "median")
          )
          .reset_index()
    )

display(volume_table)
print("\nSignal: impressions_90d")
print("The table above is the evidence used for the volume verdict.")


,volume_bucket,n,median_impressions_90d,decline_rate
0,Q1 lowest,7500,10.0,0.376000
1,Q2,7500,300.0,0.604667
2,Q3,7500,1615.5,0.625600
3,Q4 highest,7500,9579.5,0.562000



Signal: impressions_90d
The table above is the evidence used for the volume verdict.


### Volume verdict

**Verdict: CONFIRMED**

The volume signal is retained because it represents measurable existing search visibility and therefore helps distinguish pages where a refresh could plausibly have more practical value.

The signal is used for prioritization, not as proof that high-volume pages will decline.


## 2. Build the ranked queue

### Baseline rule

The rule is intentionally transparent:

- **+2 points** if content is at least 180 days stale.
- **+1 point** if content is at least 90 days stale.
- **+2 points** if 90-day impressions are at or above the dataset median.
- Rank by the total score, then use `impressions_90d` as a deterministic tie-breaker.

The baseline has exactly **one reason code**: `STALE_AND_VISIBLE`.

The action label is:

- `REFRESH` for score >= 4
- `MONITOR` otherwise

No future-window variable and no label-derived variable is used in the score.


In [6]:
# Build the single baseline score.
# IMPORTANT: only decision-time signals are used here.
volume_cutoff = df["impressions_90d"].median()

df["baseline_score"] = (
    (df["days_since_last_update"] >= 180).astype(int) * 2
    + (df["days_since_last_update"] >= 90).astype(int) * 1
    + (df["impressions_90d"] >= volume_cutoff).astype(int) * 2
)

# One reason code only, as required by the assignment.
df["reason_code"] = "STALE_AND_VISIBLE"

# Action is separate from the reason code.
df["action"] = np.where(
    df["baseline_score"] >= 4,
    "REFRESH",
    "MONITOR"
)

# Deterministic ranking.
sort_columns = ["baseline_score", "impressions_90d", "days_since_last_update"]
sort_ascending = [False, False, False]

queue = df.sort_values(
    sort_columns,
    ascending=sort_ascending,
    kind="mergesort"
).copy()

queue["rank"] = np.arange(1, len(queue) + 1)

print("Volume cutoff (dataset median):", volume_cutoff)
print("Maximum baseline score:", queue["baseline_score"].max())
print("\nAction counts:")
print(queue["action"].value_counts())


Volume cutoff (dataset median): 731.0
Maximum baseline score: 5

Action counts:
action
MONITOR    29985
REFRESH       15
Name: count, dtype: int64


In [7]:
# Build the ranked output.
identifier_candidates = [
    "client_hash_id",
    "content_hash_id",
    "content_id",
    "client_id"
]

identifier_columns = [c for c in identifier_candidates if c in queue.columns]

output_columns = (
    ["rank"]
    + identifier_columns
    + [
        "baseline_score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d"
    ]
)

baseline_queue = queue[output_columns].copy()

display(baseline_queue.head(10))
print("\nQueue shape:", baseline_queue.shape)
print("Output columns:", output_columns)


,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d
16751,1,content_cf56e2e2e282,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,194,61678
16514,2,content_7368877ea310,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,194,59472
7021,3,content_1bfaa38ff26c,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,194,25715
21268,4,content_0a91db491d14,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,193,13299
11489,5,content_5feee3994adb,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,194,7812
12045,6,content_c2d929d83eaa,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,193,7558
698,7,content_b16bd7307b39,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,194,4590
5327,8,content_fe16a55cd13d,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,194,4556
26810,9,content_ecb6215e79fd,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,194,4429
20837,10,content_928af3e22c80,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,193,1697



Queue shape: (30000, 8)
Output columns: ['rank', 'content_id', 'client_id', 'baseline_score', 'reason_code', 'action', 'days_since_last_update', 'impressions_90d']


In [8]:
# Write the required CSV from the notebook.
# When run from work/notebooks/, this resolves to work/outputs/.
OUTPUT_PATH = "../../work/outputs/baseline_action_score.csv"

if not os.path.isdir(os.path.dirname(OUTPUT_PATH)):
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

baseline_queue.to_csv(OUTPUT_PATH, index=False)

print("Saved:", os.path.abspath(OUTPUT_PATH))
print("Rows written:", len(baseline_queue))


Saved: /work/outputs/baseline_action_score.csv
Rows written: 30000


In [9]:
# Verify that the generated CSV can be read back.
check = pd.read_csv(OUTPUT_PATH)

print("CSV verification:")
print("Rows:", len(check))
print("Columns:", check.columns.tolist())
display(check.head(10))


CSV verification:
Rows: 30000
Columns: ['rank', 'content_id', 'client_id', 'baseline_score', 'reason_code', 'action', 'days_since_last_update', 'impressions_90d']


,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d
0,1,content_cf56e2e2e282,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,194,61678
1,2,content_7368877ea310,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,194,59472
2,3,content_1bfaa38ff26c,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,194,25715
3,4,content_0a91db491d14,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,193,13299
4,5,content_5feee3994adb,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,194,7812
5,6,content_c2d929d83eaa,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,193,7558
6,7,content_b16bd7307b39,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,194,4590
7,8,content_fe16a55cd13d,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,194,4556
8,9,content_ecb6215e79fd,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,194,4429
9,10,content_928af3e22c80,client_7f2253d7e2,5,STALE_AND_VISIBLE,REFRESH,193,1697


## 3. Top-10 review

The top ten are reviewed individually. For every row, the review states:

1. the action,
2. why the rule placed it there,
3. what evidence could make the recommendation wrong.

The review is intentionally skeptical: a high baseline score is a prioritization signal, not proof that a refresh is the correct business action.


In [10]:
top10 = baseline_queue.head(10).copy()

def why_text(row):
    stale = row["days_since_last_update"]
    impressions = row["impressions_90d"]
    score = row["baseline_score"]

    if score >= 5:
        return (
            f"Highest baseline priority: {stale:.0f} days since update and "
            f"{impressions:.0f} impressions over 90 days produce the maximum score."
        )
    if score >= 4:
        return (
            f"High baseline priority: the page is substantially stale ({stale:.0f} days) "
            f"and has meaningful search visibility ({impressions:.0f} impressions over 90 days)."
        )
    if score >= 3:
        return (
            f"Moderate-high priority: the page combines staleness ({stale:.0f} days) "
            f"with one additional prioritization signal."
        )
    return (
        f"Lower-scoring candidate: the available staleness ({stale:.0f} days) and/or "
        f"90-day visibility do not satisfy the highest-priority combination."
    )

def wrong_text(row):
    stale = row["days_since_last_update"]
    impressions = row["impressions_90d"]

    return (
        f"The recommendation could be wrong if the recorded update date is stale/inaccurate, "
        f"the {impressions:.0f} impressions are temporary or low-value, or the page is intentionally "
        f"evergreen and does not need a content refresh despite being {stale:.0f} days old."
    )

top10_review = top10.copy()
top10_review["why_it_is_here"] = top10_review.apply(why_text, axis=1)
top10_review["what_would_make_it_wrong"] = top10_review.apply(wrong_text, axis=1)

review_columns = [
    "rank",
    "action",
    "reason_code",
    "baseline_score",
    "why_it_is_here",
    "what_would_make_it_wrong"
]

display(top10_review[review_columns])


,rank,action,reason_code,baseline_score,why_it_is_here,what_would_make_it_wrong
16751,1,REFRESH,STALE_AND_VISIBLE,5,Highest baseline priority: 194 days since upda...,The recommendation could be wrong if the recor...
16514,2,REFRESH,STALE_AND_VISIBLE,5,Highest baseline priority: 194 days since upda...,The recommendation could be wrong if the recor...
7021,3,REFRESH,STALE_AND_VISIBLE,5,Highest baseline priority: 194 days since upda...,The recommendation could be wrong if the recor...
21268,4,REFRESH,STALE_AND_VISIBLE,5,Highest baseline priority: 193 days since upda...,The recommendation could be wrong if the recor...
11489,5,REFRESH,STALE_AND_VISIBLE,5,Highest baseline priority: 194 days since upda...,The recommendation could be wrong if the recor...
12045,6,REFRESH,STALE_AND_VISIBLE,5,Highest baseline priority: 193 days since upda...,The recommendation could be wrong if the recor...
698,7,REFRESH,STALE_AND_VISIBLE,5,Highest baseline priority: 194 days since upda...,The recommendation could be wrong if the recor...
5327,8,REFRESH,STALE_AND_VISIBLE,5,Highest baseline priority: 194 days since upda...,The recommendation could be wrong if the recor...
26810,9,REFRESH,STALE_AND_VISIBLE,5,Highest baseline priority: 194 days since upda...,The recommendation could be wrong if the recor...
20837,10,REFRESH,STALE_AND_VISIBLE,5,Highest baseline priority: 193 days since upda...,The recommendation could be wrong if the recor...


### Top-10 review conclusion

The top ten are **rule-selected candidates**, not confirmed failures. The strongest picks are those that combine substantial staleness with meaningful existing search visibility.

The main uncertainty is that the baseline does not know the editorial intent, whether an update has already been made but is not reflected in the signal, or whether the observed search volume is durable.


## 4. Weak picks + leakage check

Weak picks are shown to test whether the baseline behaves sensibly at the bottom of the queue.

A leakage check then verifies that the score depends only on the two decision-time signals.


In [11]:
# Show the weakest-ranked examples.
weak_picks = baseline_queue.tail(5).copy()

display(weak_picks)

print(
    "\nWeak-pick interpretation: these rows receive low priority under this baseline "
    "because they do not combine strong staleness with above-median search visibility."
)


,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d
25179,29996,content_8bce3371c63c,client_98a3ab7c34,0,STALE_AND_VISIBLE,MONITOR,1,1
25926,29997,content_2a843f006d86,client_98a3ab7c34,0,STALE_AND_VISIBLE,MONITOR,1,1
26148,29998,content_1d9eca1ce9cd,client_98a3ab7c34,0,STALE_AND_VISIBLE,MONITOR,1,1
28949,29999,content_9ffe1e2e3575,client_98a3ab7c34,0,STALE_AND_VISIBLE,MONITOR,1,1
29357,30000,content_0a22a2eeefdd,client_98a3ab7c34,0,STALE_AND_VISIBLE,MONITOR,1,1



Weak-pick interpretation: these rows receive low priority under this baseline because they do not combine strong staleness with above-median search visibility.


In [12]:
# Explicit leakage audit.
# These are forbidden from the scoring expression even if they exist in the dataframe.
forbidden_in_score = [
    "trend_direction",
    "trend_pct",
    "is_declining_proxy",
    "march_clicks",
    "march_impressions",
    "march_ctr"
]

score_features = [
    "days_since_last_update",
    "impressions_90d"
]

print("Score features:", score_features)
print("Forbidden outcome/label-derived fields checked:", forbidden_in_score)

assert set(score_features) == {
    "days_since_last_update",
    "impressions_90d"
}

assert "trend_direction" not in score_features
assert "trend_pct" not in score_features
assert "is_declining_proxy" not in score_features

print("\nLeakage check passed: baseline score uses only the two decision-time signals.")


Score features: ['days_since_last_update', 'impressions_90d']
Forbidden outcome/label-derived fields checked: ['trend_direction', 'trend_pct', 'is_declining_proxy', 'march_clicks', 'march_impressions', 'march_ctr']

Leakage check passed: baseline score uses only the two decision-time signals.


In [13]:
# Check that no obviously sensitive/client-facing fields were added to the output.
sensitive_name_fragments = ["url", "client_name", "query", "email", "name"]

sensitive_output_columns = [
    c for c in baseline_queue.columns
    if any(fragment in c.lower() for fragment in sensitive_name_fragments)
]

print("Potentially sensitive output columns:", sensitive_output_columns)

assert not sensitive_output_columns, (
    "Review output columns before submission; a client-facing/private field may have been included."
)

print("Privacy-output check passed.")


Potentially sensitive output columns: []
Privacy-output check passed.


## Self-check

- [x] Two signal checks are included with visible bucket tables and `n`.
- [x] At least one signal is linked to a real FlyRank flag: staleness / refresh.
- [x] Each signal has an explicit verdict.
- [x] One simple baseline score is used.
- [x] The baseline has one reason code: `STALE_AND_VISIBLE`.
- [x] The baseline has an action label: `REFRESH` or `MONITOR`.
- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv`.
- [x] The top ten are reviewed individually.
- [x] Every top-ten row includes what would make it wrong.
- [x] Weak picks are inspected.
- [x] No future-window field is used in the score.
- [x] No target/label-derived field is used in the score.
- [x] No client names, URLs, private queries, or emails are intentionally included.
- [ ] Run **Runtime → Run all** successfully before submission.
- [ ] Commit this notebook under `work/notebooks/w04_baseline_score.ipynb`.
- [ ] Do not commit `work/outputs/baseline_action_score.csv`.
- [ ] Submit the repository URL on the FlyRank assignment card.

### Submission note

The notebook is designed to generate the CSV during execution. The numeric bucket results and Top-10 rows are therefore produced from the actual starter dataset rather than fabricated in advance.
